# Examples of `snews_pt` usage

Last updated: 11/12/2023 <br>
This notebook contains the most basic publication and subscription examples.

## Subscription

In [ ]:
from snews_pt.snews_sub import Subscriber

Subscriber().subscribe()

---
# Publication

Here we create a message to which we pass the following arguments; <br>
`detector_name`, `machine_time`, `neutrino_time_utc`, `p_val`, `p_values`,`t_bin_width` <br>
As described in [Message Schema Section](https://snews-publishing-tools.readthedocs.io/en/latest/user/command_line_interface.html) of the documentation. The tiers are decided automatically by the scripts based on the input arguments passed. <br>

Here the argument `neutrino_time_utc` indicates that the message should be sent to "Coincidence Tier" for coincidence checks with the other experiments. <br>

Similarly, the input arguments; `p_values` and `t_bin_width` indicates that the message should go to "Significance Tier" for combined significance computations. For more please refer to the documentations.

In [ ]:
import os
from datetime import UTC, datetime
from snews import messages
from snews_pt.messages import Publisher

test_time = datetime.now(UTC)
print(test_time)

## Constructing messages

The method `messages.create_messages` handles the creation of messages in the right format (defined in in the [snews-data-formats](https://github.com/SNEWS2/snews-data-formats) package) and assigning the right type of Message Tier. Run `snews_pt message-schema` for more info.

In [ ]:
print('CoincidenceTierMessage fields:')
print(messages.get_fields(getattr(messages,"CoincidenceTierMessage")))
print('--------------------------------')
print('SignificanceTierMessage fields:')
print(messages.get_fields(getattr(messages,"SignificanceTierMessage")))


In [ ]:
# Simple coincidence message

snews_message_coin = messages.create_messages(
    detector_name="LZ",
    machine_time_utc=test_time,
    neutrino_time_utc=test_time,
    is_test=True,
)
snews_message_coin

In [ ]:
# Significance tier message

snews_message_sig = messages.create_messages(
    detector_name="LZ",
    machine_time_utc=test_time,
    neutrino_time_utc=test_time,
    p_val=0.0,
    p_values=[0.0007, 0.0008, 0.0009],
    t_bin_width_sec=0.07,
    is_test=True,
)
snews_message_sig

In [ ]:
for msg in snews_message_sig:
    print(type(msg).__name__)
    print(msg.model_dump())
    print('---------------------')

In [ ]:
# Example of an invalid message

snews_message_err = messages.create_messages(
    machine_time_utc=test_time,
    is_test=True,
)

## Picking a topic and publishing messages

In [ ]:
snews_message_coin

In [ ]:
firedrill = False

topic = (
    os.getenv("FIREDRILL_OBSERVATION_TOPIC")
    if firedrill
    else os.getenv("OBSERVATION_TOPIC")
)
print("Publishing to:", topic)


In [ ]:
# define a publisher object with the right topic
publisher = Publisher(kafka_topic=topic)

# add the messages to the publisher object
for msg in snews_message_coin:
    publisher.add_message(msg)

# inspect queue
publisher.message_queue

In [ ]:
# send the messages

publisher.send(verbose=2)